# Car Evaluation Dataset Classification
**Author:** Rim Zaher  
**GitHub Repository:** https://github.com/rimzaher-code/car-data-classification

---

## 1. Project Overview & Data Ingestion

### Overview:
In this project, we evaluate machine learning classification algorithms using the **Car Evaluation Dataset** from the UCI Machine Learning Repository. The goal is to predict a vehicle's acceptability rating (`unacc`, `acc`, `good`, `vgood`) based on six structural and buying attributes.

### Objectives:
* Ingest and inspect the raw dataset structure and target class distributions.
* Preprocess categorical input features and target labels into numerical formats.
* Train and evaluate a baseline **Decision Tree Classifier**.
* Train and evaluate ensemble methods (**Random Forest** and **Extra Trees**) to compare classification performance.

In [1]:
# Import pandas for data manipulation and tabular inspection
import pandas as pd

# Direct URL to the UCI Car Evaluation dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"

# Define human-readable column names based on dataset documentation
columns = [
    "buying", "maint", "doors", "persons",
    "lug_boot", "safety", "class"
]

# Ingest the dataset without headers and apply explicit column names
df = pd.read_csv(url, names=columns)

# Display dataset dimensions (rows, columns)
print("Dataset shape (rows, columns):", df.shape)

# Display the first five rows to verify correct loading
print("\nFirst 5 rows of the dataset:")
display(df.head())

# Inspect target class distribution to check for class imbalance
print("\nClass distribution in target feature ('class'):")
print(df["class"].value_counts())

Dataset shape (rows, columns): (1728, 7)

First 5 rows of the dataset:


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc



Class distribution in target feature ('class'):
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


## 2. Data Preprocessing & Train-Test Split

### Data Preparation Steps:
1. **Feature-Target Separation:** We separate the input features (`X`) from the target variable (`y`).
2. **Categorical Encoding:** Since all 6 input features are ordinal strings (e.g., `low`, `med`, `high`), we use `OrdinalEncoder` to convert them to numerical values.
3. **Target Encoding:** We apply `LabelEncoder` to convert the target classes (`acc`, `good`, `unacc`, `vgood`) into integer labels.
4. **Stratified Splitting:** We partition the dataset into an 80% training set and a 20% testing set using `stratify=y_encoded` to preserve the original class proportions across both splits.

In [2]:
# Import scikit-learn preprocessing and dataset splitting utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

# Separate independent feature columns from the target variable
X = df.drop(columns=["class"])
y = df["class"]

# Initialize OrdinalEncoder to convert categorical features to numeric integers
encoder = OrdinalEncoder()
X_encoded = pd.DataFrame(
    encoder.fit_transform(X),
    columns=X.columns
)

# Initialize LabelEncoder to encode the categorical target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Perform an 80/20 train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

# Verify the size of the training and testing subsets
print("Training records count:", len(X_train))
print("Testing records count:", len(X_test))
print("Target classes order:", list(label_encoder.classes_))

Training records count: 1382
Testing records count: 346
Target classes order: ['acc', 'good', 'unacc', 'vgood']


## 3. Baseline Model: Decision Tree Classifier

### Model Details:
We train a baseline **DecisionTreeClassifier** with hyperparameter constraints (`max_depth=6`, `min_samples_leaf=3`) to prevent overfitting. Decision trees evaluate split criteria at each node to partition data into pure decision regions.

### Evaluation Metrics:
We evaluate the model on the unseen test set using:
* **Accuracy Score:** Overall percentage of correct predictions.
* **Classification Report:** Detailed precision, recall, and F1-score for each car rating class.
* **Confusion Matrix:** Breakdown of actual vs. predicted classification counts.

In [3]:
# Import evaluation metrics and Decision Tree Classifier from scikit-learn
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Instantiate DecisionTreeClassifier with depth constraints
tree_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=3,
    random_state=42
)

# Train the decision tree model on the training set
tree_model.fit(X_train, y_train)

# Generate predictions on the holdout test set
tree_predictions = tree_model.predict(X_test)

# Calculate and display test set accuracy
dt_accuracy = accuracy_score(y_test, tree_predictions)
print("Decision Tree Accuracy:", round(dt_accuracy, 4))

# Output comprehensive classification report per class
print("\nClassification Report:")
print(classification_report(
    y_test,
    tree_predictions,
    labels=range(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

# Output Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, tree_predictions))

Decision Tree Accuracy: 0.8931

Classification Report:
              precision    recall  f1-score   support

         acc       0.69      0.95      0.80        77
        good       0.00      0.00      0.00        14
       unacc       1.00      0.92      0.96       242
       vgood       0.76      1.00      0.87        13

    accuracy                           0.89       346
   macro avg       0.61      0.72      0.66       346
weighted avg       0.88      0.89      0.88       346

Confusion Matrix:
[[ 73   0   0   4]
 [ 14   0   0   0]
 [ 19   0 223   0]
 [  0   0   0  13]]


## 4. Ensemble Methods: Random Forest & Extra Trees

### Ensemble Overview:
To improve upon single decision tree performance, we train two ensemble algorithms:
1. **Random Forest Classifier:** Builds an ensemble of bootstrap decision trees using random feature subsets at each split to reduce variance.
2. **Extra Trees Classifier (Extremely Randomized Trees):** Introduces additional randomness by choosing cut-points completely at random rather than searching for the optimal threshold.

In [4]:
# Import Ensemble models from scikit-learn
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# 1. Instantiate and train Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# 2. Instantiate and train Extra Trees Classifier
et_model = ExtraTreesClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42
)
et_model.fit(X_train, y_train)
et_predictions = et_model.predict(X_test)

# Construct a summary table comparing overall accuracy across all 3 models
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Extra Trees"],
    "Accuracy": [
        accuracy_score(y_test, tree_predictions),
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, et_predictions)
    ]
})

# Format and display sorted model comparison table
results["Accuracy"] = results["Accuracy"].round(4)
display(results.sort_values("Accuracy", ascending=False))

,Model,Accuracy
1,Random Forest,0.9827
2,Extra Trees,0.9682
0,Decision Tree,0.8931


## 5. Results & Comparative Discussion

### Summary of Model Performance:
* **Random Forest** achieved the highest test accuracy (**98.27%**), significantly outperforming the single decision tree. By averaging predictions across 200 trees, it effectively mitigated variance and generalization errors.
* **Extra Trees** achieved a strong second-place test accuracy (**96.82%**), proving that randomized decision thresholds remain effective for this categorical dataset.
* **Decision Tree Baseline** achieved **89.31%** test accuracy. While simple and interpretable, it suffered from lower recall on minority classes (such as `good` rating).

### Business Application & Conclusion:
For a car rental or dealership evaluation system, **Random Forest** is the optimal choice due to its superior accuracy across all four vehicle acceptability tiers. However, because dataset class imbalance (`unacc` makes up the majority of records) affects metrics, decision-makers should inspect precision-recall tradeoffs alongside overall accuracy.